# Dataset Viewer

Side-by-side slice viewer for **HuashanMyo** and **MyosegmenTUM**.

**How to use:**
1. Run all cells.
2. Select a dataset and subject/stack from the dropdowns.
3. Drag the slice slider.

In [ ]:
import glob, os, re, pathlib
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import SimpleITK as sitk
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, VBox, HBox
from IPython.display import display

EVAL_DIR = pathlib.Path('.')

In [ ]:
# ── Discover available stacks for each dataset ────────────────────────────────

def discover_huashanmyo(root):
    """Return {label: (water_path, fat_path, seg_path)} for HuashanMyo."""
    stacks = {}
    for w in sorted((root / 'Water').glob('THIGH_*_0001.nii.gz')):
        stem  = w.name.replace('_0001.nii.gz', '')   # THIGH_001
        fat   = root / 'Fat'   / f'{stem}_0000.nii.gz'
        seg   = root / 'Label' / f'{stem}.nii.gz'
        stacks[stem] = (w, fat if fat.exists() else None,
                           seg if seg.exists() else None)
    return stacks


def discover_myosegmentum(root):
    """Return {label: (water_path, fat_path, seg_path)} for MyosegmenTUM."""
    stacks = {}
    for w in sorted(root.glob('*/ImageData/*_WATER/*_WATER_stack*.nii')):
        m = re.match(r'(.+?)_WATER_stack(\d+)\.nii$', w.name)
        if not m:
            continue
        subj, stack = m.group(1), m.group(2)
        label = f'{subj}_stack{stack}'
        fat   = w.parent.parent / f'{subj}_FATFRACTION' / f'{subj}_FATFRACTION_stack{stack}.nii'
        seg   = root / subj / 'SegmentationMasks' / f'combined_gt_stack{stack}.mha'
        stacks[label] = (w, fat if fat.exists() else None,
                            seg if seg.exists() else None)
    return stacks


DATASETS = {}
hm_root  = EVAL_DIR / 'HuashanMyo'
myo_root = EVAL_DIR / 'myosegmenTUM'

if hm_root.exists():
    DATASETS['HuashanMyo'] = discover_huashanmyo(hm_root)
    print(f'HuashanMyo: {len(DATASETS["HuashanMyo"])} stacks')

if myo_root.exists():
    DATASETS['MyosegmenTUM'] = discover_myosegmentum(myo_root)
    print(f'MyosegmenTUM: {len(DATASETS["MyosegmenTUM"])} stacks')

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def load_norm(path):
    """Load a NIfTI/MHA file and normalise to [0, 1]."""
    arr = sitk.GetArrayFromImage(sitk.ReadImage(str(path))).astype(float)
    lo, hi = arr.min(), arr.max()
    return (arr - lo) / (hi - lo + 1e-8)


def load_label(path):
    """Load a segmentation mask as integer array."""
    return sitk.GetArrayFromImage(sitk.ReadImage(str(path))).astype(int)


def label_overlay(seg_arr, n_labels=None):
    """Build (H, W, 4) RGBA overlay from a 2-D integer label slice."""
    labels = [l for l in np.unique(seg_arr) if l != 0]
    if not labels:
        return np.zeros((*seg_arr.shape, 4))
    n = max(labels) + 1
    cmap = plt.colormaps['tab20'].resampled(max(n, 2))
    rgba = np.zeros((*seg_arr.shape, 4))
    for lbl in labels:
        r, g, b, _ = cmap(lbl - 1)
        rgba[seg_arr == lbl] = [r, g, b, 0.5]
    return rgba


_cache = {}

def get_data(dataset, stack_label):
    key = (dataset, stack_label)
    if key not in _cache:
        water_p, fat_p, seg_p = DATASETS[dataset][stack_label]
        water = load_norm(water_p)
        fat   = load_norm(fat_p)   if fat_p else None
        seg   = load_label(seg_p)  if seg_p else None
        _cache[key] = (water, fat, seg)
    return _cache[key]

In [ ]:
# ── Widgets ───────────────────────────────────────────────────────────────────

dataset_dd = Dropdown(
    options=list(DATASETS),
    description='Dataset:',
    layout=widgets.Layout(width='280px'),
)
stack_dd = Dropdown(
    options=[],
    description='Stack:',
    layout=widgets.Layout(width='380px'),
)
slice_sl = IntSlider(
    min=0, max=1, step=1, value=0,
    description='Slice:',
    layout=widgets.Layout(width='500px'),
)
show_seg_cb = widgets.Checkbox(
    value=True, description='Show segmentation',
    indent=False, layout=widgets.Layout(width='200px'),
)
out = widgets.Output()


def render(dataset, stack_label, slice_idx, show_seg):
    if not stack_label:
        return
    try:
        water, fat, seg = get_data(dataset, stack_label)
    except Exception as e:
        with out:
            out.clear_output(wait=True)
            print(f'Error: {e}')
        return

    panels = []
    panels.append(('Water', water[slice_idx], None))
    if fat is not None:
        panels.append(('Fat fraction', fat[slice_idx], None))
    if show_seg and seg is not None:
        panels.append(('Segmentation', water[slice_idx],
                        label_overlay(seg[slice_idx])))

    n = len(panels)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 6))
    if n == 1:
        axes = [axes]

    for ax, (title, img, overlay) in zip(axes, panels):
        ax.imshow(img, cmap='gray', origin='lower')
        if overlay is not None:
            ax.imshow(overlay, origin='lower')
        ax.set_title(title, fontsize=11)
        ax.axis('off')

    fig.suptitle(f'{dataset}  —  {stack_label}  —  slice {slice_idx}', fontsize=11)
    plt.tight_layout()
    with out:
        out.clear_output(wait=True)
        plt.show()


def _rerender(*_):
    render(dataset_dd.value, stack_dd.value, slice_sl.value, show_seg_cb.value)


def on_dataset_change(change):
    _cache.clear()
    stacks = list(DATASETS[change['new']])
    stack_dd.options = stacks
    stack_dd.value   = stacks[0] if stacks else None
    if stacks:
        water, _, _ = get_data(change['new'], stacks[0])
        slice_sl.max   = water.shape[0] - 1
        slice_sl.value = water.shape[0] // 2
    _rerender()


def on_stack_change(change):
    if change['new']:
        water, _, _ = get_data(dataset_dd.value, change['new'])
        slice_sl.max   = water.shape[0] - 1
        slice_sl.value = water.shape[0] // 2
    _rerender()


dataset_dd.observe(on_dataset_change, names='value')
stack_dd.observe(on_stack_change, names='value')
slice_sl.observe(lambda _: _rerender(), names='value')
show_seg_cb.observe(lambda _: _rerender(), names='value')

# Initial load
first_dataset = list(DATASETS)[0]
stacks = list(DATASETS[first_dataset])
stack_dd.options = stacks
stack_dd.value   = stacks[0]
water, _, _ = get_data(first_dataset, stacks[0])
slice_sl.max   = water.shape[0] - 1
slice_sl.value = water.shape[0] // 2
render(first_dataset, stacks[0], slice_sl.value, show_seg_cb.value)

display(VBox([
    HBox([dataset_dd, stack_dd, show_seg_cb]),
    slice_sl,
    out,
]))